# TUGAS AKHIR 03 (TA-03) – Data-Driven Monte Carlo Simulation
**Topik:** Estimasi Risiko Klaim Asuransi (digit akhir NIM 4–5)
**Dataset:** Medical Cost Personal Datasets (Kaggle) – fitur `charges`
**Pertanyaan bisnis:** Jika ada 100 pasien baru, berapa probabilitas total klaim melebihi anggaran perusahaan?

**Nama:** Daegal Fauzan Kadafi | **NIM:** 301240005

## 0. Import Library & Seed

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 301240005          # seed = NIM
np.random.seed(SEED)

## 1. Eksplorasi Data
### 1.1 Load Data
Letakkan `data.csv` (Medical Cost Personal Datasets) di folder yang sama dengan notebook (di Colab: unggah ke panel Files).

In [ ]:
CSV = "data.csv"
if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    print("Data dimuat:", df.shape)
else:
    # FALLBACK DEMO (bukan data asli) agar notebook tetap bisa dijalankan
    print("PERINGATAN: data.csv tidak ditemukan -> memakai data sintetis untuk demo!")
    df = pd.DataFrame({"age": np.random.randint(18, 65, 1338),
                       "charges": np.random.lognormal(9.1, 0.9, 1338)})
df.head()

### 1.2 Data Cleaning
Kolom yang dipakai sebagai variabel acak: **`charges`** (biaya klaim per pasien).

In [ ]:
print("Missing value:\n", df.isna().sum())
print("Duplikat:", df.duplicated().sum())
df = df.dropna(subset=["charges"]).drop_duplicates()

# Outlier ekstrem: aturan 3 x IQR
q1, q3 = df["charges"].quantile([0.25, 0.75]); iqr = q3 - q1
batas_atas = q3 + 3 * iqr
n_out = (df["charges"] > batas_atas).sum()
print(f"Batas atas outlier ekstrem = {batas_atas:,.0f}; jumlah outlier = {n_out}")

REMOVE_OUTLIER = True   # sesuai modul: hapus outlier ekstrem. (False = pertahankan klaim besar yang nyata)
if REMOVE_OUTLIER:
    df = df[df["charges"] <= batas_atas].copy()
print("Data bersih:", df.shape)

### 1.3 Parameter Extraction (Mean & Std Dev) + Histogram

In [ ]:
charges = df["charges"].values
mu, sigma = charges.mean(), charges.std(ddof=1)

# Parameter lognormal (dari log data) -> cocok untuk data miring ke kanan
log_c = np.log(charges)
mu_log, sd_log = log_c.mean(), log_c.std(ddof=1)

from scipy import stats
print(f"Mean (μ)       : {mu:,.2f}")
print(f"Std Dev (σ)    : {sigma:,.2f}")
print(f"Min / Max      : {charges.min():,.2f} / {charges.max():,.2f}")
print(f"Skewness       : {stats.skew(charges):.3f}  (>0 = miring ke kanan)")
print(f"Lognormal      : μ_log={mu_log:.4f}, σ_log={sd_log:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(charges, bins=40, color="steelblue", edgecolor="white")
ax[0].axvline(mu, color="red", linestyle="--", label=f"Mean = {mu:,.0f}")
ax[0].set_title("Histogram Charges (data asli)"); ax[0].set_xlabel("charges"); ax[0].legend()
ax[1].hist(log_c, bins=40, color="orange", edgecolor="white")
ax[1].set_title("Histogram log(Charges)"); ax[1].set_xlabel("log(charges)")
plt.tight_layout(); plt.show()

**Interpretasi distribusi:** Histogram `charges` **miring kuat ke kanan** (banyak klaim kecil, sedikit klaim sangat besar), bukan lonceng. Distribusi Normal akan menghasilkan biaya negatif dan salah menaksir ekor kanan. Karena itu simulasi utama memakai **distribusi Lognormal** (`np.random.lognormal`), sesuai catatan modul "atau distribusi lain yang sesuai dengan histogram". Simulasi dengan `np.random.normal(μ, σ)` tetap dijalankan sebagai pembanding.

## 2. Membangun Simulasi Monte Carlo
Satu skenario = 100 pasien baru, tiap pasien punya klaim acak. **Total klaim** = jumlah klaim 100 pasien.
Jumlah skenario: **10.000** (≥ 5.000 sesuai ketentuan).

In [ ]:
N_SIM = 10_000
N_PATIENTS = 100

def simulate_total_claims(dist="lognormal"):
    if dist == "lognormal":
        claims = np.random.lognormal(mu_log, sd_log, (N_SIM, N_PATIENTS))
    else:   # normal (pembanding), dipotong di nilai minimum data agar tidak negatif
        claims = np.clip(np.random.normal(mu, sigma, (N_SIM, N_PATIENTS)), charges.min(), None)
    return claims.sum(axis=1)

np.random.seed(SEED)
total_ln = simulate_total_claims("lognormal")
np.random.seed(SEED)
total_nm = simulate_total_claims("normal")

for nama, t in [("Lognormal (utama)", total_ln), ("Normal (pembanding)", total_nm)]:
    print(f"{nama:22s} mean={t.mean():>12,.0f}  std={t.std():>10,.0f}  "
          f"P5={np.percentile(t,5):>12,.0f}  P95={np.percentile(t,95):>12,.0f}")

## 3. Analisis Risiko & Kesimpulan
**Anggaran perusahaan** untuk 100 pasien ditetapkan = 110% dari ekspektasi total klaim (100 × μ), dibulatkan. Ubah `BUDGET` bila kasus Anda menetapkan angka lain.

$$P(Event)=\frac{\text{Jumlah skenario yang memenuhi kondisi}}{\text{Total skenario}}$$

In [ ]:
BUDGET = round(1.10 * N_PATIENTS * mu, -4)
print(f"Anggaran perusahaan (BUDGET): {BUDGET:,.0f}\n")

p_exceed = np.mean(total_ln > BUDGET)
print(f"P(total klaim > anggaran) [Lognormal] = {p_exceed:.4f}  ({p_exceed*100:.2f}%)")
print(f"P(total klaim > anggaran) [Normal]    = {np.mean(total_nm > BUDGET):.4f}")

print("\nSensitivitas terhadap besar anggaran:")
for mult in [1.0, 1.05, 1.10, 1.20, 1.30]:
    b = mult * N_PATIENTS * mu
    print(f"  Anggaran {b:>12,.0f} ({mult:.0%} dari ekspektasi) -> P(melebihi) = {np.mean(total_ln > b):.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
n, bins, patches = plt.hist(total_ln, bins=50, color="steelblue", edgecolor="white")
for b, p in zip(bins[:-1], patches):
    if b >= BUDGET:
        p.set_facecolor("tomato")          # area risiko (melebihi anggaran)
plt.axvline(BUDGET, color="black", linestyle="--", linewidth=2, label=f"Anggaran = {BUDGET:,.0f}")
plt.axvline(total_ln.mean(), color="green", linestyle=":", linewidth=2, label=f"Mean = {total_ln.mean():,.0f}")
plt.title(f"Distribusi Total Klaim 100 Pasien ({N_SIM:,} Skenario Monte Carlo)")
plt.xlabel("Total klaim"); plt.ylabel("Frekuensi")
plt.text(0.98, 0.95, f"P(melebihi anggaran) = {p_exceed:.2%}", transform=plt.gca().transAxes,
         va="top", ha="right", color="tomato", fontsize=11, fontweight="bold")
plt.legend(loc="upper left"); plt.show()

In [ ]:
print("KESIMPULAN")
print("-" * 60)
print(f"Dari {N_SIM:,} skenario, total klaim 100 pasien baru melebihi anggaran "
      f"{BUDGET:,.0f} pada {int(p_exceed*N_SIM):,} skenario.")
print(f"Probabilitas total klaim melebihi anggaran: {p_exceed:.2%}")
print(f"Ekspektasi total klaim: {total_ln.mean():,.0f} "
      f"(95% rentang: {np.percentile(total_ln,2.5):,.0f} - {np.percentile(total_ln,97.5):,.0f})")